In [16]:
import json
import re
from datetime import datetime, timedelta
from pathlib import Path


# from lecture 6
# create requirements.txt by using pip freeze in your repo

### Task 1

In [ ]:
USER = "Bruker"
BASE_DIR = Path("C:\\", "Users", USER, "Documents", "folder_for_saving", "d2_data")

def read_file(filepath: str) -> tuple(str, str):
    """
    Opens and reads file. Checking encoding type between UTF-8 and ISO-8859-1

    ARGS: file path
    
    RETURNS: tuple containing strings for file text content and encoding.
    """

    # create a path object
    path = Path(filepath)

    # tries opening with UTF-8 encoding first and with latin-1/ISO-8859-1 second
    # This should correctly decode files encoded with UTF and ISO-8859-1
    # does not always return correct encoding as some non conflicting ISO-8859-1 files
    # can be opened in UTF-8
    try:
        filedata = open(path, encoding="UTF-8").read(), "UTF-8"
        return filedata
    except:
        filedata = open(path, encoding="ISO-8859-1").read(), "ISO-8859-1"
        return filedata


# test (not sure if the generated files from generate_data_for_d2 is the same so check generation log if encoding is correct)
# mine was. Giving data and UTF-8 encoding which is correct
# print(read_file(r"C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\science_building\POS_02\register_sales.log")[1])

# # for me this should be ISO-8859-1. It is not. 
print(read_file(r"C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\science_building\POS_03\register_sales.log"))
# print(read_file(r"{BASE_DIR}\2026\science_building\POS_03\register_sales.log"))

def parse_log(filecontent: str) -> list[dict]:
    """
    Reads files line by line and checks if lines are a valid log format and then extracts
    info from that log entry.

    ARGS: Content of a register_sales.log file.

    RETURNS: A tuple list containing dictionaries where each dictionary is a valid transaction

    WARNING: Does not handle tuples from read_file
    """
    valid_sales = []
    # Set up regex pattern for the that should be matched from log
    pattern = re.compile(
        r"\[(?P<datetime>[\d\-]+\s[\w:]+)\] "
        r"\[(?P<register_id>POS_\d+)\] "
        r"item=(?P<item>\S+) "
        r"price=(?P<price>\S+) "
        r"(?P<currency>\w+) "
        r"\((?P<status>\w+)\)$",
        re.MULTILINE
    )

    # Creates a list of dictionaries where each dictionary will be a valid transaction by
    # Iterating over the lines in filecontent  
    valid_sales = [match.groupdict() for match in pattern.finditer(filecontent)]
    print(valid_sales)

# Test
parse_log(read_file(r"C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\science_building\POS_03\register_sales.log")[0])


('[2026-10-15 07:02:40] [POS_03] item=Sandwich price=7.58 EUR (SUCCESS)\n=== NETWORK SYNC OK ===\n[2026-10-15 07:04:09] [POS_03] item=Muffin price=3.17 NOK (SUCCESS)\n[2026-10-15 07:04:44] [POS_03] item=Sandwich price=7.01 EUR (REFUND)\n[2026-10-15 07:06:25] [POS_03] item=Latte price=4.54 USD (SUCCESS)\n[2026-10-15 07:08:52] [POS_03] item=Croissant price=3.49 EUR (SUCCESS)\n[2026-10-15 07:09:20] [POS_03] item=Sandwich price=7.64 NOK (SUCCESS)\n[2026-10-15 07:11:24] [POS_03] item=Cappuccino price=4.16 EUR (CANCELLED)\n[2026-10-15 07:13:18] [POS_03] item=Croissant price=4.19 USD (REFUND)\n[DEBUG] BARCODE SCANNER REBOOTED\n[2026-10-15 07:14:38] [POS_03] item=Croissant price=4.14 NOK (REFUND)\n[2026-10-15 07:14:53] [POS_03] item=Cappuccino price=4.47 USD (CANCELLED)\n[2026-10-15 07:17:40] [POS_03] item=Espresso price=3.30 EUR (REFUND)\n[2026-10-15 07:18:45] [POS_03] item=Croissant price=3.83 USD (SUCCESS)\n[2026-10-15 07:20:36] [POS_03] item=Cappuccino price=4.52 USD (SUCCESS)\n=== NETWORK

The above code should be able to sort all the given files. It will not catch files that are ambiguous which seems to be almost impossible for UTF-8 and ISO-8859-1 without external info about the file. As with the short test done above. I have not run it on the whole file set yet so I do not know if it will be able to tell a single file appart from the given set. 

### Task 2

In [ ]:
def find_log_files(path: str) -> list[Path]:
    # Convert to Path object
    folder = Path(path)

    logs = []

    # iterate over items in directory
    for child in folder.iterdir():
        if child.is_file() and str(child)[-4:] == ".log":
            logs.append(child)  # if child is a .log file add Path to logs
        elif child.is_dir():
            # if child is a directory call function and add elements to logs via extend
            logs.extend(find_log_files(child))
    return logs

# Test
print(find_log_files(r"C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026"))


def get_active_registers(configpath: str) -> set[str]:
    config = Path(configpath)



[WindowsPath('C:/Users/Bruker/Documents/folder_for_saving/d2_data/2026/library_cafe/POS_01/register_sales.log'), WindowsPath('C:/Users/Bruker/Documents/folder_for_saving/d2_data/2026/library_cafe/POS_02/register_sales.log'), WindowsPath('C:/Users/Bruker/Documents/folder_for_saving/d2_data/2026/library_cafe/POS_03/register_sales.log'), WindowsPath('C:/Users/Bruker/Documents/folder_for_saving/d2_data/2026/science_building/POS_01/register_sales.log'), WindowsPath('C:/Users/Bruker/Documents/folder_for_saving/d2_data/2026/science_building/POS_02/register_sales.log'), WindowsPath('C:/Users/Bruker/Documents/folder_for_saving/d2_data/2026/science_building/POS_03/register_sales.log'), WindowsPath('C:/Users/Bruker/Documents/folder_for_saving/d2_data/2026/student_union/POS_01/register_sales.log'), WindowsPath('C:/Users/Bruker/Documents/folder_for_saving/d2_data/2026/student_union/POS_02/register_sales.log'), WindowsPath('C:/Users/Bruker/Documents/folder_for_saving/d2_data/2026/student_union/POS_0